In [19]:
from pystac_client import Client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access

import geopandas as gpd
import joblib
import numpy as np
import odc.geo 
import pandas as pd
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier
import rioxarray 
import xarray as xr
from utils import load_data, safe_divide

In [20]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [21]:
# Create local dask cluster to improve data load time. Only run this once.
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='16GB')

# Configure S3 access. Cloud defaults is an optimisation, while requester pays is required for Landsat
configure_s3_access(cloud_defaults=True, requester_pays=True)

In [11]:
# Configure some things up front
chunks = dict(x=2048, y=2048)
datetime = "2022"

# bbox_fiji = [177.14, -18.41, 179.99, -16.0]
# bbox = bbox_fiji

bbox = [177.24, -17.95, 177.70, -17.58]
# bbox = [178.14, -17.81, 178.50, -17.4] #tailevu
bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({'geometry': [bbox_geometry]}, crs='EPSG:4326')
gdf.explore()

In [37]:
merged = load_data(bbox, chunks=dict(x=2048, y=2048), resolution=10)

In [38]:
merged

<xarray.Dataset> Size: 3GB
Dimensions:        (y: 4300, x: 5121)
Coordinates:
  * y              (y) float64 34kB -1.976e+06 -1.976e+06 ... -2.019e+06
  * x              (x) float64 41kB 3.032e+06 3.032e+06 ... 3.084e+06 3.084e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2022-01-01
Data variables: (12/25)
    blue           (y, x) uint16 44MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    green          (y, x) uint16 44MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    red            (y, x) uint16 44MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge1       (y, x) uint16 44MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge2       (y, x) uint16 44MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge3       (y, x) uint16 44MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    ...             ...
    ndwi           (y, x) float64 176MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    ndre           (y, x) float64 176MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    elevation      (y, x) float32 88MB 0.0 0.0 0.0 0.0 ... 34.86 33.89 33.89
    slope          (y, x) float32 88MB nan nan nan nan 0.0 ... nan nan nan nan
    aspect         (y, x) float32 88MB nan nan nan nan 90.0 ... nan nan nan nan
    elevation_std  (y, x) float32 88MB nan nan nan nan nan ... nan nan nan nan

## Train and predict

When you change your training data, you can re-train and predict here.

In [39]:
training_file = "training_data/mred-tdata.geojson"

tdata = gpd.read_file(training_file, bbox=bbox_geometry)
tdata = tdata.set_geometry("geometry")
tdata = tdata.to_crs(merged.rio.crs)

print(tdata.crs)
print(merged.rio.crs)
if tdata.crs != merged.rio.crs:
    tdata = tdata.to_crs(merged.rio.crs)

tdata.explore()

PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",150],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3832"]]
EPSG:3832


In [40]:
len(tdata)

321

In [41]:
tdata['lulc_code'].value_counts().sort_index()

lulc_code
1     45
2     27
3     30
4     44
5     55
6      4
7     13
8    103
Name: count, dtype: int64

In [42]:
# 1. Ensure both tables are aligned correctly using the shared 'index' column
# This prevents positional or index-label mismatches from creating NaN rows
training_array = pd.merge(tdata, pt_values, on="index", suffixes=('_tdata', '_pt'))

# 2. If tdata is a GeoDataFrame, ensure geometry is preserved from tdata
if not isinstance(training_array, gpd.GeoDataFrame):
    training_array = gpd.GeoDataFrame(training_array, geometry=tdata.geometry, crs=tdata.crs)

# 3. Reproject the spatial GeoDataFrame to WGS84 (EPSG:4326) safely
training_array = training_array.to_crs(4326)

# 4. Drop unwanted metadata, coordinate, and duplicate columns cleanly
cols_to_drop = [
    "y_tdata", "x_tdata", "y_pt", "x_pt", "y", "x",
    "spatial_ref", "time", "fid", "index", "lulc_class",
    "path", "geometry", "layer", "id", "Unnamed: 0"
]

# Only drop columns that actually exist in the dataframe to avoid KeyErrors
existing_cols_to_drop = [col for col in cols_to_drop if col in training_array.columns]
training_array = training_array.drop(columns=existing_cols_to_drop)

# 5. Check final shape and clean NaNs safely
print("Final training array shape:", training_array.shape)
training_array = training_array.dropna(subset=['lulc_code'])

NameError: name 'pt_values' is not defined

In [ ]:
len(training_array)

In [ ]:
training_array

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 1:]
classes = np.array(training_array)[:, 0]

model = classifier.fit(training_data, classes)


In [ ]:
joblib.dump(model, "test_model.dump")

In [ ]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(training_array.columns[1:], classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    # Format as a table to 2 decinal places
    print(f"{i[0]:<11}| {i[1]:.2f}")
